# Spectral Gauges of Displacement Covariances

Spectral Wasserstein distances replace the trace of the displacement covariance
$$
    M_\pi=\int (x-y)(x-y)^\top\,d\pi(x,y)
$$
by a monotone spectral gauge $\gamma(M_\pi)$.  This notebook compares the ordinary trace gauge with an approximate $\lambda_{\max}$ robust gauge on a small discrete problem.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from matplotlib.collections import LineCollection
from scipy.optimize import linprog
import ot

NAME = "spectral-wasserstein-gauge"
out = figure_dir(NAME)
rng = np.random.default_rng(34)

The $\lambda_{\max}$ plan is computed by minimizing the largest projected quadratic displacement over a finite set of directions.  This is a small illustrative LP, not a benchmark.

In [2]:
def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

n = 18
x = rng.normal(size=(n, 2)) @ np.diag([0.28, 0.18]) @ rot(np.deg2rad(8)).T + np.array([-0.48, 0.02])
y = np.vstack([
    rng.normal(size=(9, 2)) @ np.diag([0.20, 0.10]) @ rot(np.deg2rad(35)).T + np.array([0.70, 0.48]),
    rng.normal(size=(9, 2)) @ np.diag([0.20, 0.10]) @ rot(np.deg2rad(-35)).T + np.array([0.70, -0.48]),
])
a = b = np.ones(n) / n
C_trace = ot.dist(x, y)
P_trace = ot.emd(a, b, C_trace)

# Approximate min over couplings of max_theta E[<theta,x-y>^2].
angles = np.linspace(0, np.pi, 25, endpoint=False)
dirs = np.column_stack([np.cos(angles), np.sin(angles)])
diff = x[:, None, :] - y[None, :, :]
proj_costs = np.stack([(diff @ th)**2 for th in dirs], axis=0)
num_p = n * n
num_vars = num_p + 1
c = np.zeros(num_vars)
c[-1] = 1.0
A_eq = []
b_eq = []
for i in range(n):
    row = np.zeros(num_vars)
    row[i*n:(i+1)*n] = 1
    A_eq.append(row); b_eq.append(a[i])
for j in range(n):
    row = np.zeros(num_vars)
    row[j:num_p:n] = 1
    A_eq.append(row); b_eq.append(b[j])
A_ub = []
b_ub = []
for Cth in proj_costs:
    row = np.zeros(num_vars)
    row[:num_p] = Cth.ravel()
    row[-1] = -1
    A_ub.append(row); b_ub.append(0.0)
bounds = [(0, None)] * num_p + [(0, None)]
res = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), A_eq=np.array(A_eq), b_eq=np.array(b_eq), bounds=bounds, method="highs")
if not res.success:
    raise RuntimeError(res.message)
P_spec = res.x[:num_p].reshape(n, n)

def plan_pairs(P, max_edges=85):
    entries = [(i, j, float(P[i, j])) for i in range(n) for j in range(n) if P[i, j] > 1e-7]
    entries = sorted(entries, key=lambda z: z[2], reverse=True)[:max_edges]
    return entries

pairs_trace = plan_pairs(P_trace)
pairs_spec = plan_pairs(P_spec)
pts = np.vstack([x, y])
xlim, ylim = padded_limits(pts, pad=0.14)

def moment(P):
    M = np.einsum('ij,ijk,ijl->kl', P, diff, diff)
    return M
print('trace eigenvalues', np.linalg.eigvalsh(moment(P_trace)))
print('spectral eigenvalues', np.linalg.eigvalsh(moment(P_spec)))

trace eigenvalues [0.12335643 1.03885482]
spectral eigenvalues [0.22781809 1.03386354]


In [3]:
def decorate(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    remove_axes(ax)

def draw_plan(P, pairs, filename, color):
    fig, ax = plt.subplots(figsize=(2.35, 2.10))
    draw_transport_segments(ax, x, y, pairs, color=color, min_width=0.13, max_width=1.02, alpha_scale=0.50, zorder=1)
    draw_point_clouds(ax, x, y, base_size=DIRAC_MARKER_SIZE * 0.74)
    decorate(ax)
    save_pdf(fig, out / filename, pad_inches=0.055)
    plt.close(fig)

def draw_geodesic(P, filename):
    fig, ax = plt.subplots(figsize=(2.35, 2.10))
    entries = [(i, j, float(P[i, j])) for i in range(n) for j in range(n) if P[i, j] > 1e-7]
    mmax = max(m for _, _, m in entries)
    for k, t in enumerate([0.0, 0.33, 0.66, 1.0]):
        pts_t = []
        sizes = []
        for i, j, m in entries:
            pts_t.append((1-t)*x[i] + t*y[j])
            sizes.append(DIRAC_MARKER_SIZE * (0.26 + 1.05 * np.sqrt(m / mmax)))
        pts_t = np.asarray(pts_t)
        ax.scatter(pts_t[:,0], pts_t[:,1], s=sizes, marker="o", color=interp_color(t), edgecolor="none", linewidth=0, alpha=0.50 if 0 < t < 1 else 0.72, zorder=2+k)
    decorate(ax)
    save_pdf(fig, out / filename, pad_inches=0.055)
    plt.close(fig)

draw_plan(P_trace, pairs_trace, "trace-coupling.pdf", VIOLET)
draw_plan(P_spec, pairs_spec, "lambda-max-coupling.pdf", ORANGE)
draw_geodesic(P_trace, "trace-geodesic.pdf")
draw_geodesic(P_spec, "lambda-max-geodesic.pdf")
# Backward-compatible aliases for older gallery cards.
draw_geodesic(P_trace, "round.pdf")
draw_geodesic(P_spec, "anisotropic.pdf")
draw_plan(P_spec, pairs_spec, "rank-one.pdf", ORANGE)